In [7]:
import pandas as pd
import numpy as np

# First I am loading my full dataset
df = pd.read_csv(
    "WC2026_Interceptions_Analytical_datasets.csv",
    encoding="cp1252"
)

# I am checking the first few rows
print(df.head())

   Rank              Player Position   Squad  Age  Born  90s  Tackles  \
0     1    Brenden Aaronson       MF     USA   25  2000  0.8        1   
1     2      Thelo Aasgaard       MF  Norway   24  2002  1.0        1   
2     3    Hamza Abdelkarim       FW   Egypt   18  2008  0.7        0   
3     4  Hossam Abdelmaguid       DF   Egypt   25  2001  0.7        0   
4     5  Mohamed Abdelmonem       DF   Egypt   27  1999  0.2        0   

   Interceptions (Int) Tournament Stage  Interceptions per 90 mins  \
0                    0         Knockout                   0.000000   
1                    3         Knockout                   3.000000   
2                    0         Knockout                   0.000000   
3                    1         Knockout                   1.428571   
4                    0         Knockout                   0.000000   

  Eligible Defender FBref Player ID  Unnamed: 13  Unnamed: 14  Unnamed: 15  \
0                No        5bc43860          NaN          NaN 

In [8]:
# I am removing the empty unnamed columns from the dataset
# These columns were created during the CSV export and are not useful for my analysis

df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# I am checking the columns again to make sure they are removed
print(df.columns)

Index(['Rank', 'Player', 'Position', 'Squad', 'Age', 'Born', '90s', 'Tackles',
       'Interceptions (Int)', 'Tournament Stage', 'Interceptions per 90 mins',
       'Eligible Defender', 'FBref Player ID'],
      dtype='str')


In [9]:
# I am checking the first few rows after removing the unnecessary columns

print(df.head())

   Rank              Player Position   Squad  Age  Born  90s  Tackles  \
0     1    Brenden Aaronson       MF     USA   25  2000  0.8        1   
1     2      Thelo Aasgaard       MF  Norway   24  2002  1.0        1   
2     3    Hamza Abdelkarim       FW   Egypt   18  2008  0.7        0   
3     4  Hossam Abdelmaguid       DF   Egypt   25  2001  0.7        0   
4     5  Mohamed Abdelmonem       DF   Egypt   27  1999  0.2        0   

   Interceptions (Int) Tournament Stage  Interceptions per 90 mins  \
0                    0         Knockout                   0.000000   
1                    3         Knockout                   3.000000   
2                    0         Knockout                   0.000000   
3                    1         Knockout                   1.428571   
4                    0         Knockout                   0.000000   

  Eligible Defender FBref Player ID  
0                No        5bc43860  
1                No        c88a28b9  
2                No       

In [10]:
# I am checking how many rows and columns are left after cleaning

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 1039
Columns: 13


In [11]:
# Now I am keeping only the players who are eligible defenders
# This removes forwards, midfielders, goalkeepers and defenders with no valid playing time

df_defenders = df[df["Eligible Defender"] == "Yes"].copy()

# I am checking how many defenders are left
print("Eligible defenders:", df_defenders.shape[0])

# I am checking the first few rows
print(df_defenders.head())

Eligible defenders: 349
   Rank              Player Position         Squad  Age  Born  90s  Tackles  \
3     4  Hossam Abdelmaguid       DF         Egypt   25  2001  0.7        0   
4     5  Mohamed Abdelmonem       DF         Egypt   27  1999  0.2        0   
5     6            Ali Abdi       DF       Tunisia   32  1993  3.0        8   
6     7     Saud Abdulhamid       DF  Saudi Arabia   26  1999  3.0        9   
7     8   Abdulla Abdullaev       DF    Uzbekistan   28  1997  2.0        2   

   Interceptions (Int)        Tournament Stage  Interceptions per 90 mins  \
3                    1                Knockout                   1.428571   
4                    0                Knockout                   0.000000   
5                    4  Group Stage Eliminated                   1.333333   
6                    2  Group Stage Eliminated                   0.666667   
7                    3  Group Stage Eliminated                   1.500000   

  Eligible Defender FBref Player ID  


In [12]:
# I am checking how many defenders are in each tournament stage

print(df_defenders["Tournament Stage"].value_counts())

Tournament Stage
Knockout                  238
Group Stage Eliminated    111
Name: count, dtype: int64


In [13]:
# Now I am checking the playing time of the defenders
# This helps me find players who only played for a very short time

print(df_defenders["90s"].describe())

count    349.000000
mean       2.549570
std        1.736948
min        0.100000
25%        1.000000
50%        2.500000
75%        3.700000
max        8.300000
Name: 90s, dtype: float64


In [14]:
# I am checking how many defenders played less than one full 90 minutes

low_playing_time = df_defenders[df_defenders["90s"] < 1.0]

print("Defenders with less than 1.0 90s:", len(low_playing_time))

Defenders with less than 1.0 90s: 69


In [15]:
# I am looking at the defenders with very low playing time
# because their per-90 interception rate may be less reliable

print(
    low_playing_time[
        ["Player", "Squad", "90s", "Interceptions (Int)", "Interceptions per 90 mins"]
    ]
)

                     Player        Squad  90s  Interceptions (Int)  \
3        Hossam Abdelmaguid        Egypt  0.7                    1   
4        Mohamed Abdelmonem        Egypt  0.2                    0   
10    Mohammad Abu Hasheesh       Jordan  0.1                    0   
12             Mo Abualnadi       Jordan  0.8                    0   
67    Fredrik André Bjørkan       Norway  0.5                    2   
...                     ...          ...  ...                  ...   
988        Francis de Vries  New Zealand  0.2                    1   
989           Luka Vuškovi?      Croatia  0.7                    1   
1006            Axel Witsel      Belgium  0.4                    0   
1020           Manaf Younis         Iraq  0.8                    1   
1033             David Zima      Czechia  0.1                    0   

      Interceptions per 90 mins  
3                      1.428571  
4                      0.000000  
10                     0.000000  
12                     

In [16]:
# I am removing defenders who played less than one full 90 minutes
# because very low playing time can make the per-90 rate unreliable

df_final = df_defenders[df_defenders["90s"] >= 1.0].copy()

# I am checking how many defenders are left after this step

print("Defenders remaining:", df_final.shape[0])

# I am checking the two tournament groups again

print(df_final["Tournament Stage"].value_counts())

Defenders remaining: 280
Tournament Stage
Knockout                  190
Group Stage Eliminated     90
Name: count, dtype: int64


In [17]:
# I am checking the main variable again after removing low playing-time players

print(df_final["Interceptions per 90 mins"].describe())

count    280.000000
mean       1.048491
std        0.734570
min        0.000000
25%        0.555556
50%        1.000000
75%        1.379870
max        5.000000
Name: Interceptions per 90 mins, dtype: float64


In [18]:
# I am saving my cleaned defender dataset as a new CSV file

df_final.to_csv(
    "WC2026_Cleaned_Defenders_Final.csv",
    index=False
)

print("Cleaned CSV file saved successfully.")

Cleaned CSV file saved successfully.


In [19]:
import os

print(os.getcwd())

c:\Users\youha\Desktop\HIT140_Assignment


In [20]:
print(os.path.exists("WC2026_Cleaned_Defenders_Final.csv"))

True
